# GraphRAG (2024)
---
[[paper]](https://arxiv.org/abs/2403.04126)
GraphRAG = Graph-based Retrieval Augmented Generation

GraphRAG – это инновационный фреймворк для Retrieval Augmented Generation (RAG), разработанный Microsoft Research в 2024 году, который использует знания, извлеченные и структурированные в виде графа, для улучшения процесса поиска контекста и качества генерации Large Language Models (LLM).

### Контекст
Традиционные подходы к RAG, такие как Dense Passage Retrieval (DPR (2020)) или Contriever (2021), хорошо работают для извлечения релевантных текстовых фрагментов (chunks) по запросу. Однако они сталкиваются с ограничениями, когда речь идет о сложных запросах, требующих синтеза информации из нескольких документов или понимания неявных связей. Например, ответы на вопросы, требующие многоходовой логики (multi-hop reasoning), или обобщение информации о сущностях, распределенных по разным частям базы знаний, часто не поддаются линейному поиску по текстовым фрагментам. Это приводит к неполным ответам, галлюцинациям и неспособности LLM глубоко "понять" контекст. Простое увеличение контекстного окна LLM (как в GPT-4 Turbo (2023) с 128k токенов) также не решает проблему структурной нехватки и ведет к росту затрат и потенциальному снижению качества из-за "потерянного в середине" эффекта.

### Идея
Основная идея GraphRAG заключается в трансформации неструктурированных текстовых данных в структурированный **граф знаний (Knowledge Graph)**. Этот граф затем используется не только для извлечения релевантной информации, но и для **создания согласованного и обогащенного контекста** для LLM, что позволяет выполнять многоходовое рассуждение и эффективно использовать реляционные знания. Вместо того чтобы просто передавать LLM набор фрагментов, GraphRAG строит динамический подграф, отражающий структуру и связи запрошенной информации, и затем синтезирует из него связный контекст.

### Задача
GraphRAG решает задачу улучшенной генерации ответов с использованием внешних данных, особенно для сложных запросов, требующих глубокого понимания контекста, синтеза информации из различных источников и выполнения многоходовых рассуждений в больших коллекциях документов.

### Существующие альтернативы
На момент появления GraphRAG существовали следующие основные подходы к RAG:
*   **Стандартные RAG-фреймворки:** Например, те, что используют Dense Retrieval (DPR (2020), Contriever (2021)) для поиска наиболее похожих текстовых фрагментов (passages) и последующей их подачи в LLM. Их архитектурное отличие заключалось в фокусировке на семантическом сходстве линейных текстовых фрагментов, игнорируя при этом их взаимосвязи.
*   **Naive Graph-based RAG:** Некоторые ранние попытки использовали графы для простого поиска сущностей или фильтрации, но не для глубокого, управляемого LLM обхода графа и синтеза информации внутри RAG-процесса.
*   **Увеличение контекстного окна LLM:** Подходы, которые пытались решить проблему, предоставляя LLM максимально большой контекст. Это является наивным решением, так как не учитывает релевантность частей контекста и приводит к проблемам масштабирования, вычислительным затратам и эффекту "потерянного в середине", когда LLM не может эффективно использовать информацию из длинных контекстов.
*   **Методы переранжирования (Re-ranking):** Такие как ColBERT (2020) или MonoT5 (2020), которые улучшают качество извлеченных документов, но не решают структурных ограничений линейного текста.
*   **Гибридный RAG:** Объединение методов Sparse (BM25) и Dense Retrieval, но все еще оперирующий с плоскими документами без учета их внутренних связей.

### Архитектура
GraphRAG состоит из нескольких взаимосвязанных модулей:

1.  **Модуль создания графа (Graph Creation Module):**
    *   **Вход:** Неструктурированные текстовые документы.
    *   **Процесс:** Использует LLM (или специализированные NLP-модели) для извлечения сущностей (entities), отношений (relationships) между ними и ключевых концепций из документов. Дополнительно может извлекать краткие резюме или аннотации для каждого фрагмента текста.
    *   **Выход:** **Знаниевый граф (Knowledge Graph)**, где узлы (nodes) представляют сущности, концепции, события или исходные текстовые фрагменты (document chunks), а ребра (edges) обозначают связи между ними (например, "является частью", "содержит", "связан с", "автор"). Этот граф может быть обогащен метаданными и эмбеддингами для узлов и ребер.

2.  **Модуль понимания запроса (Query Understanding Module):**
    *   **Вход:** Пользовательский запрос.
    *   **Процесс:** Использует LLM для анализа запроса, идентификации ключевых сущностей, отношений и вывода возможных путей обхода графа, необходимых для ответа на запрос. Может включать переформулирование запроса или разбивку его на подзапросы.
    *   **Выход:** Structured query representation или набор инструкций для обхода графа.

3.  **Модуль обхода/извлечения графа (Graph Traversal/Retrieval Module):**
    *   **Вход:** Структурированное представление запроса и знаниевый граф.
    *   **Процесс:** Выполняет интеллектуальный обход графа на основе понимания запроса. Это может включать:
        *   **Многоходовой поиск (Multi-hop search):** Нахождение путей между сущностями.
        *   **Извлечение подграфов:** Идентификация наиболее релевантного подграфа, содержащего ключевую информацию.
        *   **Расширение контекста:** Включение в подграф соседних узлов и ребер для обогащения контекста.
        *   Методы могут включать Random Walk with Restart, персонализированный PageRank, или обход графа, управляемый LLM.
    *   **Выход:** **Релевантный подграф (Relevant Subgraph)**, содержащий узлы и ребра, критически важные для ответа на запрос.

4.  **Модуль синтеза контекста (Context Synthesis Module):**
    *   **Вход:** Релевантный подграф.
    *   **Процесс:** Агрегирует информацию из узлов и ребер подграфа, переформулируя ее в связный, удобочитаемый текст. Это может включать:
        *   **Резюмирование:** Использование LLM для сжатия информации из множества узлов.
        *   **Упорядочивание:** Структурирование информации в логическом порядке.
        *   **Добавление связей:** Явное описание отношений между сущностями в тексте.
    *   **Выход:** **Синтезированный контекст (Synthesized Context)** – высококачественный, структурированный текст, готовый для подачи в финальную LLM.

5.  **Модуль генерации (Generation Module):**
    *   **Вход:** Синтезированный контекст и оригинальный пользовательский запрос.
    *   **Процесс:** Подаёт синтезированный контекст вместе с запросом в финальную LLM для генерации ответа.
    *   **Выход:** Сгенерированный ответ LLM.

### Алгоритм обучения
GraphRAG как фреймворк не "обучается" в традиционном смысле end-to-end с использованием обратного распространения ошибки. Вместо этого он использует несколько **предварительно обученных LLM** и других моделей NLP для выполнения своих задач. "Обучение" в контексте GraphRAG подразумевает:
*   **Fine-tuning компонентных LLM:** Отдельные LLM, используемые в модулях (например, для извлечения сущностей/отношений, суммаризации подграфов, переформулирования запросов), могут быть дополнительно fine-tuned на специфических датасетах для повышения их производительности в рамках конкретной предметной области.
*   **Настройка промтов и эвристик:** Инженерная оптимизация промтов для LLM и правил для обхода графа, агрегации информации и других шагов процесса.

### Алгоритм инференса
1.  Пользователь отправляет **запрос**.
2.  **Модуль понимания запроса** анализирует запрос и преобразует его в инструкции для работы с графом.
3.  **Модуль обхода/извлечения графа** использует эти инструкции для поиска наиболее релевантного **подграфа** в знаниевом графе.
4.  **Модуль синтеза контекста** агрегирует и структурирует информацию из подграфа, создавая **синтезированный контекст**.
5.  **Модуль генерации** подает синтезированный контекст вместе с оригинальным запросом в LLM для получения окончательного **ответа**.

### Результаты
Microsoft Research утверждает, что GraphRAG демонстрирует **значительные улучшения** по сравнению со стандартными RAG-подходами, особенно на задачах, требующих глубокого понимания и синтеза информации:
*   **Улучшение фактологической консистентности:** GraphRAG способен значительно сократить количество галлюцинаций. На синтетическом датасете, разработанном для проверки синтеза информации, было достигнуто **сокращение галлюцинаций примерно на 30%**.
*   **Повышение релевантности и полноты ответов:** Модель показала более высокие результаты на бенчмарках вопросов-ответов, требующих многоходового рассуждения (например, HotpotQA), благодаря способности обходить и синтезировать информацию из связанных, но разрозненных частей базы знаний.
*   **Улучшение качества генерации:** На задачах суммаризации и ответа на вопросы GraphRAG показал более высокие метрики ROUGE и BLEU, что свидетельствует о лучшей связности и информативности сгенерированных текстов.
*   **Качественная разница:** GraphRAG эффективно решает качественно новый тип проблем, связанный с глубоким, реляционным рассуждением, что было труднодостижимо для предыдущих RAG-методов, оперирующих только с линейными фрагментами текста.

## 📝 Критический анализ

```markdown
# GraphRAG (2024)
---
[[paper]](https://arxiv.org/abs/2403.04126)
GraphRAG = Graph-based Retrieval Augmented Generation

GraphRAG – это фреймворк от Microsoft Research для Retrieval Augmented Generation (RAG), использующий граф знаний для улучшения поиска контекста и генерации Large Language Models (LLM).

### Контекст
Традиционные RAG, такие как Dense Passage Retrieval (DPR (2020)) и Contriever (2021), ограничены в сложных запросах, требующих синтеза информации из нескольких документов. Простое увеличение контекстного окна LLM не решает проблему структурной нехватки и ведет к росту затрат.

### Идея
GraphRAG преобразует текстовые данные в **граф знаний (Knowledge Graph)**, который используется для извлечения информации и создания обогащенного контекста для LLM, позволяя выполнять многоходовое рассуждение.

### Задача
GraphRAG улучшает генерацию ответов для сложных запросов, требующих глубокого понимания контекста и синтеза информации из различных источников.

### Существующие альтернативы
- **Стандартные RAG-фреймворки:** Используют Dense Retrieval, игнорируя взаимосвязи.
- **Naive Graph-based RAG:** Применяют графы для простого поиска сущностей.
- **Увеличение контекстного окна LLM:** Не учитывает релевантность частей контекста.
- **Методы переранжирования (Re-ranking):** Улучшают качество извлеченных документов, но не решают структурных ограничений.
- **Гибридный RAG:** Комбинирует Sparse и Dense Retrieval, но игнорирует внутренние связи.

### Архитектура
GraphRAG состоит из модулей:

1. **Graph Creation Module:**
   - Извлекает сущности и отношения из текстов, создавая **граф знаний**.
   
2. **Query Understanding Module:**
   - Анализирует запрос и формирует инструкции для обхода графа.

3. **Graph Traversal/Retrieval Module:**
   - Выполняет интеллектуальный обход графа, извлекая **релевантный подграф**.

4. **Context Synthesis Module:**
   - Агрегирует информацию из подграфа в связный текст.

5. **Generation Module:**
   - Генерирует ответ на основе синтезированного контекста.

### Алгоритм обучения
GraphRAG использует **предварительно обученные LLM** и другие модели NLP. Обучение включает fine-tuning LLM и настройку промтов.

### Алгоритм инференса
1. Анализ запроса.
2. Поиск релевантного подграфа.
3. Синтез контекста.
4. Генерация ответа.

### Результаты
GraphRAG улучшает фактологическую консистентность, снижая галлюцинации на 30%. Повышает релевантность и полноту ответов, улучшая метрики ROUGE и BLEU.

<img src="img/img.png" width=500>
```

## 💻 Пример кода

Иллюстративный Python пример, демонстрирующий основные концепции:

In [ ]:
# Пример реализации основных концепций GraphRAG

# Импорт необходимых библиотек
import networkx as nx
from transformers import pipeline

# 1. Модуль создания графа (Graph Creation Module)
# Создадим простой граф знаний из неструктурированного текста

# Пример неструктурированного текста
documents = [
    "Alice is a researcher at University X. She works on AI.",
    "Bob is a professor at University Y. He collaborates with Alice.",
    "University X and University Y are partners in AI research."
]

# Создаем граф
knowledge_graph = nx.Graph()

# Извлечение сущностей и отношений (упрощенно)
entities = {
    "Alice": {"type": "Person"},
    "Bob": {"type": "Person"},
    "University X": {"type": "Organization"},
    "University Y": {"type": "Organization"}
}

relationships = [
    ("Alice", "University X", "works at"),
    ("Bob", "University Y", "works at"),
    ("Alice", "Bob", "collaborates with"),
    ("University X", "University Y", "partners with")
]

# Добавляем узлы и ребра в граф
for entity, attributes in entities.items():
    knowledge_graph.add_node(entity, **attributes)

for source, target, relation in relationships:
    knowledge_graph.add_edge(source, target, relation=relation)

# 2. Модуль понимания запроса (Query Understanding Module)
# Используем LLM для анализа запроса

query = "Who collaborates with researchers at University X?"

# Используем предварительно обученную модель для анализа запроса
query_analyzer = pipeline("ner", model="dbmdz/bert-large-cased-finetuned-conll03-english")

# Извлечение ключевых сущностей из запроса
query_entities = query_analyzer(query)
print("Query Entities:", query_entities)

# 3. Модуль обхода/извлечения графа (Graph Traversal/Retrieval Module)
# Находим релевантный подграф

# Функция для поиска путей в графе
def find_collaborators(graph, start_entity):
    collaborators = []
    for neighbor in graph.neighbors(start_entity):
        if graph[start_entity][neighbor]['relation'] == 'collaborates with':
            collaborators.append(neighbor)
    return collaborators

# Извлечение подграфа
relevant_subgraph = find_collaborators(knowledge_graph, "Alice")
print("Collaborators with Alice:", relevant_subgraph)

# 4. Модуль синтеза контекста (Context Synthesis Module)
# Агрегируем информацию из подграфа

# Пример простого синтеза
synthesized_context = f"Alice collaborates with {', '.join(relevant_subgraph)}."

# 5. Модуль генерации (Generation Module)
# Генерация ответа на основе синтезированного контекста

# Используем LLM для генерации ответа
generator = pipeline("text-generation", model="gpt2")

# Генерируем ответ
response = generator(f"Based on the context: {synthesized_context} Answer the query: {query}", max_length=50)
print("Generated Response:", response[0]['generated_text'])

# Этот пример иллюстрирует основные этапы GraphRAG:
# - Создание графа знаний из текстовых данных
# - Понимание запроса и извлечение ключевых сущностей
# - Обход графа для нахождения релевантного подграфа
# - Синтез контекста из подграфа
# - Генерация ответа с использованием LLM
```

Этот код демонстрирует основные концепции GraphRAG, такие как создание графа знаний, понимание запроса, извлечение релевантного подграфа, синтез контекста и генерация ответа. Он иллюстрирует, как GraphRAG может использовать структурированные графы для улучшения процесса поиска и генерации ответов.